# Mirage: the two external models on the fresh IMDb reviews (Kaggle, T4 GPU)

For the confirmatory check (G8, D53): Hugging Face's DistilBERT and RoBERTa-large on **10,000 IMDb
reviews never used before** (`imdb_fresh`) and their first + last sentences (`imdb_fresh_edges`).
Predictions only, no training, nothing to upload. Everything else in the check runs on the laptop.

1. Settings: **GPU T4 x1** and **Internet = On** (check Internet again before saving: it can switch
   off when a session restarts, and the run then fails within a minute).
2. **Save Version → Save & Run All**. About 25–35 minutes. You can close the laptop.
3. When it's done, download `fresh_outputs.zip` from the output panel and unzip it into the repo
   root (`C:\claude\Mirage`). You should get 4 files in `results/predictions/`.

In [ ]:
REPO_URL = "https://github.com/AditeyaAItronics/mirage.git"

In [ ]:
import os, subprocess
os.chdir("/kaggle/working") if os.path.isdir("/kaggle/working") else None
if os.path.basename(os.getcwd()) == "mirage":
    os.chdir("..")
if os.path.isdir("mirage"):  # throwaway clone: always move it to the latest commit
    subprocess.run(["git", "-C", "mirage", "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", "mirage", "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, "mirage"], check=True)
os.chdir("mirage")
print("code version:", subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!pip install -q -e ".[dl]"
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU!")

In [ ]:
# The fresh sample is rebuilt from fixed seeds, exactly as on the laptop.
!python -m mirage prepare
# run as a separate process: the package was installed in this session, so importing it
# here could fail until the kernel restarts
!python -c "from mirage.data import prepare_fresh; print(prepare_fresh().to_string(index=False))"
import hashlib, pandas as pd
fresh = pd.read_parquet("artifacts/data/imdb_fresh.parquet")
digest = hashlib.sha256("
".join(fresh["text"]).encode()).hexdigest()[:16]
print(len(fresh), "reviews; first ids:", fresh["item_id"].head(3).tolist(), "; text fingerprint:", digest)

In [ ]:
!python -m mirage predict-external --datasets imdb_fresh imdb_fresh_edges

In [ ]:
import glob, zipfile
out = "/kaggle/working/fresh_outputs.zip" if os.path.isdir("/kaggle/working") else "fresh_outputs.zip"
files = [f for m in ("roberta_large", "distilbert_hf") for f in glob.glob(f"results/predictions/{m}__*__imdb_fresh*.parquet")]
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f)
print(out, os.path.getsize(out) // 2**10, "KB")
for f in files:
    p = pd.read_parquet(f)
    print(f, len(p), "rows, accuracy", round(p["correct"].mean(), 4))